# Agente de Investigación con Búsqueda de Bing

## Descripción

En este ejercicio, crearás un **agente de IA que utiliza la Búsqueda de Bing** para responder preguntas sobre una amplia variedad de temas. Este agente aprovechará el poder de la búsqueda en tiempo real para proporcionar información actualizada y relevante.

### Casos de Uso
- **Investigación general**: Obtener información sobre cualquier tema.
- **Verificación de hechos**: Comprobar la veracidad de la información.
- **Resumen de noticias**: Obtener las últimas noticias sobre un tema.

### Herramientas Utilizadas
- **Búsqueda de Bing**: El agente utilizará la Búsqueda de Bing para encontrar información en la web.
- **Azure AI Agents SDK**: Para crear y gestionar el agente.

## Prerrequisitos

Asegúrate de que tu archivo `.env` en la raíz del repositorio contiene las variables `PROJECT_ENDPOINT`, `MODEL_DEPLOYMENT_NAME` y `BING_CONNECTION_NAME` correctas. Este archivo se reutiliza en todos los laboratorios.

### Cargar librerías y variables de entorno

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.ai.agents.models import BingGroundingTool

load_dotenv(find_dotenv(usecwd=True))
project_endpoint = os.getenv("PROJECT_ENDPOINT")
model_deployment = os.getenv("MODEL_DEPLOYMENT_NAME")
bing_connection_name = os.getenv("BING_CONNECTION_NAME")

### Conectar al Cliente de Proyectos de IA (AIProjectClient)

Ahora, nos conectaremos al proyecto de Azure AI Foundry usando las credenciales de Azure.

In [ ]:
print("Connecting to Azure AI Project Client...")
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential()
)
print("AI Project client connected successfully")

### Crear un Agente con la Herramienta de Búsqueda de Bing

Para que la herramienta de Búsqueda de Bing esté disponible para tu agente, utiliza una conexión para inicializar la herramienta y adjuntarla al agente.

In [ ]:
print("Initializing Bing Grounding tool...")
bing = BingGroundingTool(connection_id=bing_connection_name)
print("Bing Grounding tool initialized")

with project_client:
    print("Creating agent...")
    agent = project_client.agents.create_agent(
        model=model_deployment,
        name="bing-researcher",
        instructions="Eres un agente de investigación útil que responde preguntas utilizando la Búsqueda de Bing.",
        tools=bing.definitions,
    )
    print(f"Agent created: {agent.name} (id: {agent.id})")

### Crear un Hilo (Thread) para la Conversación

Iniciamos un hilo donde se ejecutará la sesión de chat con el agente.

In [ ]:
print("Creating thread...")
thread = project_client.agents.threads.create()
print(f"Thread created (id: {thread.id})")

### Definir la Lógica para Enviar Prompts y Procesar Respuestas

Esta función encapsula la lógica para enviar un mensaje al agente, procesarlo y obtener la respuesta.

In [ ]:
def send_prompt_to_agent(prompt):
    print(f"Sending prompt: '{prompt}'")
    message = project_client.agents.messages.create(
        thread_id=thread.id,
        role="user",
        content=prompt,
    )
    print(f"User message created (thread: {thread.id})")

    print("Processing run...")
    run = project_client.agents.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent.id,
    )
    print(f"Run completed with status: {run.status}")

    if run.status == "failed":
        print(f"Run failed: {run.last_error}")
    else:
        print("Retrieving agent response...")
        messages = project_client.agents.messages.list(thread_id=thread.id)
        for message in messages:
            if message.role == 'assistant':
                print("Agent response:")
                print(f"{message.content}")

print("Function 'send_prompt_to_agent' defined successfully")

In [ ]:
user_prompt = "¿Cuál es el estado actual de la exploración espacial?"

send_prompt_to_agent(user_prompt)

### Limpieza

Finalmente, elimina el agente para liberar recursos.

In [ ]:
print(f"Deleting agent (id: {agent.id})...")
project_client.agents.delete_agent(agent.id)
print("Agent deleted successfully")